In [37]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

# Pick some data file...
df = pd.read_csv('./splits/midwest-big-train.csv')

In [39]:
df.count()

rating    23999999
text      23999704
type      23999999
dtype: int64

In [40]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def preprocess_text(text):
    # Maybe there's a better way to do this, but it gets hung up on reviews that are just integers 
    # I don't know why someone would leave such a review but there are a few
    text = str(text) 

    # Make lowercase and remove puncuation 
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    
    return word_tokenize(text)

# This takes in a dictionary, which should be influenced by the review data
def load_glove_embeddings(glove_file, dictionary):
    """Loads GloVe embeddings from a file."""
    embeddings = {}
    
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in dictionary:
                vector = np.asarray(values[1:], dtype='float32')
                embeddings[word] = vector

    return embeddings
    
## Get stopwords from nltk
stop_words = list(stopwords.words('english')) 

In [44]:
def proportional_sample(df, col_name, sample_size):
    """
    Samples a DataFrame proportionally to the distribution of values in a column,
    without replacement.

    Args:
        df (pd.DataFrame): The DataFrame to sample from.
        col_name (str): The name of the column to use for proportional sampling.
        sample_size (int): The desired size of the sampled DataFrame.

    Returns:
        pd.DataFrame: The sampled DataFrame.
    """

    value_counts = df[col_name].value_counts(normalize=True)
    sampled_counts = (value_counts * sample_size).round().astype(int)

    sampled_dfs = []
    for value, count in sampled_counts.items():
        subset = df[df[col_name] == value].sample(n=min(count, len(df[df[col_name] == value])), replace=False) #sample without replacement, take the minimum of the count or the amount of that value in the df.
        sampled_dfs.append(subset)

    sampled_df = pd.concat(sampled_dfs)
    return sampled_df


sample_size = 1000000
df_for_dict = proportional_sample(df, 'rating', sample_size)

In [47]:
# create dict from review text
# Use "df_for_dict" sampled dictionary if the df becomes too large
# otherwise this can take awhile
df_dict = dict()
for review in df_for_dict['text']:

    # Grab words from review
    word_list = [word for word in preprocess_text(review) if word not in stop_words]
    for word in word_list:
        if word in df_dict:
            df_dict[word] += 1

        else: 
            df_dict[word] = 1

In [48]:
# sort by most common appearing words
# df_dict = {key: value for key, value in df_dict.items() if value > 1}
sorted_dict = list(dict(sorted(df_dict.items(), key=lambda item: item[1], reverse=True)))

# Create vocab as dict word:idx ordered by most common appearance
vocab = dict()
i = 1
vocab_len_max = 20000

for word in sorted_dict:
    vocab[word] = i
    if i == vocab_len_max:
        break

    else:
        i += 1


In [49]:
# This will create the embedding matrix
def create_embedding_matrix(glove_embeddings, vocab):
    embedding_dim = glove_len
    num_embeddings = len(vocab) + 1
    embedding_matrix = np.zeros((num_embeddings, embedding_dim))
    
    for word, idx in vocab.items():
        if word not in glove_embeddings:
            continue
            
        embedding_vector = glove_embeddings.get(word)
        if embedding_vector is not None:
            embedding_matrix[idx] = embedding_vector
            
    return torch.tensor(embedding_matrix, dtype=torch.float32)

## Set directory of glove file and load the embeddings
glove_file = './glove/glove.6B.50d.txt' 
glove_embeddings = load_glove_embeddings(glove_file, vocab)

## Set dim of GloVe vectors (we'll use the 50d)
## 'food' is a safe word to pick since it's usually the most common
glove_len = len(glove_embeddings['food'])

# Create embedding matrix
embedding_matrix = create_embedding_matrix(glove_embeddings, vocab)

In [50]:
embedding_matrix.size()

torch.Size([20001, 50])

In [51]:
# Check that the matrix is the right thing
for index, (key, value) in enumerate(vocab.items()):
        if index < 5:
            print(value, key)
            print()

            print('embedding mat')
            print(embedding_matrix[value])
            
            print('glove emb')
            print(glove_embeddings[key])
            print()
        else:
            break

1 food

embedding mat
tensor([ 0.4722, -0.4455, -0.5183, -0.2682,  0.4443, -0.2511, -0.9928, -0.9020,
         1.8729,  0.0391,  0.1428,  0.0749,  1.0543, -0.3203,  1.0722,  0.4432,
         0.0099,  0.1575,  0.5140, -0.7767,  0.9240,  0.0110,  0.5882,  0.2308,
        -0.3428, -0.8844, -0.3149,  0.1266,  1.1445,  0.6077,  3.4344,  0.6356,
        -0.1383,  0.2804, -0.1618,  0.7754, -0.4989,  0.4602,  0.9180,  0.2901,
         0.0688,  0.5998,  0.5397, -0.0618,  1.2975,  0.9232, -0.8094,  0.3493,
         0.3393,  0.2550])
glove emb
[ 0.47222   -0.44545   -0.51833   -0.26818    0.44427   -0.25108
 -0.99282   -0.90198    1.8729     0.039081   0.14284    0.074878
  1.0543    -0.3203     1.0722     0.44323    0.0099484  0.15754
  0.51399   -0.77668    0.924      0.010958   0.58815    0.23078
 -0.34281   -0.88444   -0.31492    0.12661    1.1445     0.60775
  3.4344     0.63561   -0.13832    0.28045   -0.16181    0.77541
 -0.49888    0.4602     0.91799    0.29007    0.06884    0.59978
  0.5

In [52]:
# Converts a string to vocab indices, and pads/truncates all reviews to be 25 tokens
def stringerizer(sentence, max_length=25):
    words = preprocess_text(sentence)
    ids = []
    for word in words:
        # If a word doesn't appear in either of these just ignore it
        if (word not in glove_embeddings) or (word not in vocab):
            continue

        # 
        else: ids.append(vocab[word])

    # 
    return ids[:max_length] + [0] * (max_length - len(ids))

In [ ]:
# Grab review, rating pairs

X = [] # Stringerized text
y = [] # Labels 

for row in df.itertuples():
    # This takes awhile
    X.append(stringerizer(row[2]))
    
    # CrossEntropy wants the true values formatted starting with 0
    label = row[1]
    # Labels 1--5

    # Default - keep 1-5 ratings (This is not very accurate)
    # y.append(label-1) 
    
    # Try a neg/neutral/pos model
    if label in {1, 2}:
        y.append(0)
    elif label in {3, 4}:
        y.append(1)
    else:
        y.append(2)

# torchify the data
X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

In [ ]:
print('X', X.size())
print('y', y.size())

# let's save the data here
torch.save(X, 'X-midwest.pt')
torch.save(y, 'y-midwest.pt')

In [ ]:
# and save the embedding matrix
import pickle

def save_dictionary_pickle(dictionary, filename):
    """Saves a dictionary to a pickle file."""
    with open(filename, 'wb') as f:  # 'wb' for binary write
        pickle.dump(dictionary, f)

save_dictionary_pickle(embedding_matrix, 'em-midwest.pkl')